In [1]:
import tensorflow as tf
import os


BASE_DIR = '/kaggle/input/datasets/xhlulu/140k-real-and-fake-faces/real_vs_fake/real-vs-fake'

train_dir = os.path.join(BASE_DIR, 'train')
valid_dir = os.path.join(BASE_DIR, 'valid')
test_dir = os.path.join(BASE_DIR, 'test')


BATCH_SIZE = 16
IMG_SIZE = (224, 224) # (for EfficientNet)


train_dataset = tf.keras.utils.image_dataset_from_directory(
    train_dir,
    shuffle=True, 
    batch_size=BATCH_SIZE,
    image_size=IMG_SIZE
)


valid_dataset = tf.keras.utils.image_dataset_from_directory(
    valid_dir,
    shuffle=True,
    batch_size=BATCH_SIZE,
    image_size=IMG_SIZE
)


test_dataset = tf.keras.utils.image_dataset_from_directory(
    test_dir,
    shuffle=False, 
    batch_size=BATCH_SIZE,
    image_size=IMG_SIZE
)


class_names = train_dataset.class_names
print(f"\nClasses: {class_names}")

2026-05-24 15:47:38.410578: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1779637658.610811      23 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1779637658.665015      23 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1779637659.144302      23 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1779637659.144342      23 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1779637659.144344      23 computation_placer.cc:177] computation placer alr

Found 100000 files belonging to 2 classes.


I0000 00:00:1779637755.016808      23 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 13757 MB memory:  -> device: 0, name: Tesla T4, pci bus id: 0000:00:04.0, compute capability: 7.5
I0000 00:00:1779637755.022714      23 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:1 with 13757 MB memory:  -> device: 1, name: Tesla T4, pci bus id: 0000:00:05.0, compute capability: 7.5


Found 20000 files belonging to 2 classes.
Found 20000 files belonging to 2 classes.

Classes: ['fake', 'real']


In [2]:
# AUTOTUNE for the current system
AUTOTUNE = tf.data.AUTOTUNE

train_dataset = train_dataset.prefetch(buffer_size=AUTOTUNE)
valid_dataset = valid_dataset.prefetch(buffer_size=AUTOTUNE)
test_dataset = test_dataset.prefetch(buffer_size=AUTOTUNE)

In [3]:
from tensorflow.keras.applications import EfficientNetB0
from tensorflow.keras import layers, models, callbacks

base_model = EfficientNetB0(input_shape=(224, 224, 3), include_top=False, weights='imagenet')

# Freeze the base model 
base_model.trainable = False

# Build final model
model = models.Sequential([
    base_model,
    layers.GlobalAveragePooling2D(),
    layers.Dropout(0.5), 
    layers.Dense(1, activation='sigmoid') # probability between 0 and 1
])

print("Compile")
model.compile(optimizer='adam',
              loss='binary_crossentropy', 
              metrics=['accuracy'])

early_stopping = callbacks.EarlyStopping(
    monitor='val_loss',
    patience=3,
    restore_best_weights=True
)

print("Training")
EPOCHS = 10 

history = model.fit(
    train_dataset,
    validation_data=valid_dataset,
    epochs=EPOCHS,
    callbacks=[early_stopping]
)

# SAVE THE MODEL
model.save('/kaggle/working/deepfake_detector.keras')


16705208/16705208 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step
Compile
Training
Epoch 1/10


I0000 00:00:1779637794.545436      75 service.cc:152] XLA service 0x7c7fdc115ab0 initialized for platform CUDA (this does not guarantee that XLA will be used). Devices:
I0000 00:00:1779637794.545474      75 service.cc:160]   StreamExecutor device (0): Tesla T4, Compute Capability 7.5
I0000 00:00:1779637794.545478      75 service.cc:160]   StreamExecutor device (1): Tesla T4, Compute Capability 7.5
I0000 00:00:1779637797.004895      75 cuda_dnn.cc:529] Loaded cuDNN version 91002
2026-05-24 15:50:04.032418: E external/local_xla/xla/stream_executor/cuda/cuda_timer.cc:86] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.
2026-05-24 15:50:04.171724: E external/local_xla/xla/stream_executor/cuda/cuda_timer.cc:86] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.
2026-05-24 15:50:04.495739: E external/local_xl

6250/6250 ━━━━━━━━━━━━━━━━━━━━ 242s 34ms/step - accuracy: 0.7320 - loss: 0.5312 - val_accuracy: 0.8031 - val_loss: 0.4288
Epoch 2/10
6250/6250 ━━━━━━━━━━━━━━━━━━━━ 149s 24ms/step - accuracy: 0.7666 - loss: 0.4854 - val_accuracy: 0.8073 - val_loss: 0.4219
Epoch 3/10
6250/6250 ━━━━━━━━━━━━━━━━━━━━ 150s 24ms/step - accuracy: 0.7670 - loss: 0.4848 - val_accuracy: 0.8112 - val_loss: 0.4173
Epoch 4/10
6250/6250 ━━━━━━━━━━━━━━━━━━━━ 147s 23ms/step - accuracy: 0.7702 - loss: 0.4817 - val_accuracy: 0.8120 - val_loss: 0.4148
Epoch 5/10
6250/6250 ━━━━━━━━━━━━━━━━━━━━ 148s 24ms/step - accuracy: 0.7701 - loss: 0.4813 - val_accuracy: 0.8126 - val_loss: 0.4146
Epoch 6/10
6250/6250 ━━━━━━━━━━━━━━━━━━━━ 147s 24ms/step - accuracy: 0.7662 - loss: 0.4848 - val_accuracy: 0.8093 - val_loss: 0.4179
Epoch 7/10
6250/6250 ━━━━━━━━━━━━━━━━━━━━ 149s 24ms/step - accuracy: 0.7695 - loss: 0.4812 - val_accuracy: 0.8100 - val_loss: 0.4172
Epoch 8/10
6250/6250 ━━━━━━━━━━━━━━━━━━━━ 149s 24ms/step - accuracy: 0.7691 - lo